# 일사량 예측 프로젝트

**목표**: 기상변수로부터 일사량을 예측하는 모델 개발

**데이터**:
- KMA_weather.csv: 기상청 데이터 (학습용)
- total_iot_sensor.csv: IoT 센서 데이터 (예측용)

In [ ]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

## 1. 데이터 로드 및 기본 탐색

In [ ]:
# KMA 데이터 로드
kma_df = pd.read_csv('KMA_weather.csv', encoding='cp949')
print("=== KMA 기상청 데이터 ===")
print(f"데이터 크기: {kma_df.shape}")
print(f"컬럼 수: {len(kma_df.columns)}")
print(f"시간 범위: {kma_df['일시'].min()} ~ {kma_df['일시'].max()}")
print("\n컬럼 목록:")
for i, col in enumerate(kma_df.columns, 1):
    print(f"{i:2d}. {col}")
print("\n첫 5줄:")
kma_df.head()

In [ ]:
# IoT 데이터 로드
iot_df = pd.read_csv('total_iot_sensor.csv')
print("\n=== IoT 센서 데이터 ===")
print(f"데이터 크기: {iot_df.shape}")
print(f"컬럼 수: {len(iot_df.columns)}")
print(f"시간 범위: {iot_df['create_at'].min()} ~ {iot_df['create_at'].max()}")
print("\n컬럼 목록:")
for i, col in enumerate(iot_df.columns, 1):
    print(f"{i:2d}. {col}")
print("\n첫 5줄:")
iot_df.head()

## 2. 데이터 정제 및 특성 선택

**핵심 질문**: IoT 센서가 제공하는 변수들로 일사량을 예측할 수 있는가?

In [ ]:
# KMA 데이터에서 IoT 센서가 제공하는 변수들 매핑
print("=== IoT 센서 ↔ KMA 데이터 매핑 ===")
iot_to_kma_mapping = {
    'temperature': '기온(°C)',
    'humidity': '습도(%)',
    'pressure': '해면기압(hPa)',  # 현지기압 vs 해면기압 중 선택 필요
    'wind_speed': '풍속(m/s)',
    'rainfall_accumulation': '강수량(mm)'
}

print("IoT 센서 변수 → KMA 변수 매핑:")
for iot_var, kma_var in iot_to_kma_mapping.items():
    print(f"  {iot_var} → {kma_var}")

# KMA에서 사용할 수 있는 추가 변수들
additional_vars = ['전운량(10분위)', '지면온도(°C)']
print(f"\nKMA에서 추가로 사용할 수 있는 변수: {additional_vars}")
print("※ IoT 센서에는 없으므로 예측시 기본값 사용")

In [ ]:
# 데이터 정제: 결측치 확인
print("=== KMA 데이터 결측치 현황 ===")
target_col = '일사(MJ/m2)'
all_features = list(iot_to_kma_mapping.values()) + additional_vars

for col in [target_col] + all_features:
    if col in kma_df.columns:
        null_cnt = kma_df[col].isnull().sum()
        total = len(kma_df)
        pct = null_cnt / total * 100
        print(f"{col:15} | 결측치: {null_cnt:4d} / {total} ({pct:5.1f}%)")

# 낮시간 데이터만 필터링 (일사량은 주간에만 의미있음)
kma_df['일시'] = pd.to_datetime(kma_df['일시'])
kma_df['시간'] = kma_df['일시'].dt.hour

kma_daytime = kma_df[(kma_df['시간'] >= 6) & (kma_df['시간'] <= 18)].copy()
print(f"\n전체 데이터: {len(kma_df):,}개")
print(f"낮시간 데이터: {len(kma_daytime):,}개")

# 유효한 데이터만 선택
kma_clean = kma_daytime[[target_col] + all_features].dropna()
print(f"결측치 제거 후: {len(kma_clean):,}개")

## 3. 모델 개발

**전략**: IoT 센서가 제공하는 변수들로만 예측 모델 개발

In [ ]:
# 특성 선택: IoT 센서가 실제로 제공하는 변수들만 사용
# ※ 강수량은 결측치가 많아서 제외
iot_available_features = ['기온(°C)', '습도(%)', '해면기압(hPa)', '풍속(m/s)']

print(f"사용할 특성들: {iot_available_features}")
print(f"타겟: {target_col}")

# 데이터 준비
X = kma_clean[iot_available_features]
y = kma_clean[target_col]

print(f"\n학습 데이터 크기: {X.shape}")
print(f"일사량 범위: {y.min():.3f} ~ {y.max():.3f} MJ/m²")
print(f"일사량 평균: {y.mean():.3f} MJ/m²")

In [ ]:
# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 특성 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"학습 데이터: {X_train_scaled.shape}")
print(f"테스트 데이터: {X_test_scaled.shape}")

In [ ]:
# 모델 학습
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

# 성능 평가
train_score = model.score(X_train_scaled, y_train)
test_score = model.score(X_test_scaled, y_test)

print(f"학습 데이터 R²: {train_score:.4f}")
print(f"테스트 데이터 R²: {test_score:.4f}")

# 상세 평가
y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(((y_test - y_pred) ** 2).mean())

print(f"MAE (평균 절대 오차): {mae:.4f} MJ/m²")
print(f"RMSE (제곱근 평균 오차): {rmse:.4f} MJ/m²")

# 특성 중요도
feature_importance = pd.DataFrame({
    'feature': iot_available_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n특성 중요도:")
for idx, row in feature_importance.iterrows():
    print(f"  {row['feature']:10}: {row['importance']:.4f}")

## 4. 모델 검증 및 해석

In [ ]:
# 예측 vs 실제값 비교
plt.figure(figsize=(10, 6))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('실제 일사량 (MJ/m²)')
plt.ylabel('예측 일사량 (MJ/m²)')
plt.title(f'예측 vs 실제 (R² = {test_score:.3f})')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
errors = y_test - y_pred
plt.hist(errors, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('예측 오차 (MJ/m²)')
plt.ylabel('빈도')
plt.title(f'예측 오차 분포 (평균: {errors.mean():.3f})')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_validation.png', dpi=100, bbox_inches='tight')
plt.show()

print("모델 검증 그래프 저장: model_validation.png")

## 5. IoT 데이터에 적용

In [ ]:
# IoT 데이터 준비
iot_features = list(iot_to_kma_mapping.keys())[:4]  # 강수량 제외
iot_clean = iot_df[iot_features].copy()

# 컬럼명 변경
iot_clean = iot_clean.rename(columns={
    'temperature': '기온(°C)',
    'humidity': '습도(%)',
    'pressure': '해면기압(hPa)',
    'wind_speed': '풍속(m/s)'
})

print(f"IoT 예측 데이터 크기: {iot_clean.shape}")
print("\nIoT 데이터 샘플:")
print(iot_clean.head())

# 예측 수행
iot_scaled = scaler.transform(iot_clean)
iot_predictions = model.predict(iot_scaled)

# 결과 통합
iot_with_pred = iot_df.copy()
iot_with_pred['predicted_insolation'] = iot_predictions

# 음수 예측값 제거
iot_with_pred['predicted_insolation'] = iot_with_pred['predicted_insolation'].clip(lower=0)

print(f"\n예측 완료: {len(iot_predictions):,}개")
print(f"예측 일사량 범위: {iot_predictions.min():.3f} ~ {iot_predictions.max():.3f} MJ/m²")
print(f"예측 일사량 평균: {iot_predictions.mean():.3f} MJ/m²")

In [ ]:
# 결과 저장
output_df = iot_with_pred[['create_at', 'device_id', 'temperature', 'humidity', 
                          'wind_speed', 'pressure', 'predicted_insolation']].copy()

output_df.to_csv('iot_insolation_predictions_clean.csv', index=False)

print("결과 저장 완료: iot_insolation_predictions_clean.csv")
print(f"저장된 레코드: {len(output_df):,}개")
print(f"시간 범위: {output_df['create_at'].min()} ~ {output_df['create_at'].max()}")

# 모델 저장
import joblib
joblib.dump(model, 'insolation_model_final.pkl')
joblib.dump(scaler, 'scaler_final.pkl')

print("\n모델 저장 완료:")
print("- insolation_model_final.pkl")
print("- scaler_final.pkl")

## 6. 결론 및 다음 단계

### 현재 성능
- **R² 점수**: ?
- **MAE**: ? MJ/m²
- **RMSE**: ? MJ/m²

### 개선 방안
1. **더 많은 데이터**: 계절별, 지역별 다양성 추가
2. **시간 기반 특성**: 태양 고도각, 계절 정보 추가
3. **IoT 센서 업그레이드**: 전운량, 지면온도 센서 추가
4. **앙상블 모델**: 여러 모델 결합

### 실무 적용
- 새로운 IoT 데이터가 들어오면 `model.predict(scaler.transform(data))` 사용
- 예측 정확도는 도메인 요구사항에 따라 평가